# Baseline capability ladder on bundled ASAS-SN light curves

This notebook compares the baseline implementations in the order they were introduced:

1. global median;
2. per-camera rolling median;
3. unmasked per-camera GP;
4. the previous masked GP (camera-local masking plus late-onset consensus); and
5. the current masked GP, which additionally propagates an excursion interval when at least two same-band cameras independently corroborate it.

The first comparison uses real bundled light curves. A separate camera-partition ablation then runs both the unmasked and full masked GP paths with either one pooled GP per filter or one GP per physical camera. This keeps the V/g distinction while changing only the camera partition, so the extra per-camera complexity can be assessed directly. A stage-by-stage view then separates the stiff GP, the previous camera-local mask, newly propagated points, late-onset consensus, the final quiescent baseline, and standardized residuals. The final synthetic example demonstrates the explicitly opt-in, colour-calibrated cross-band consensus path.

For the controlled old-versus-new comparison, the notebook temporarily disables only the current code's corroborated-interval combiner. Every other implementation detail and default remains identical.

In [ ]:
from contextlib import contextmanager
from pathlib import Path
import sqlite3
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(
    (
        path for path in candidate_roots
        if (path / 'pyproject.toml').is_file()
        and (path / 'malca' / 'core' / 'baseline.py').is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError(f'Could not find the MALCA repository above {Path.cwd()}')

for path in (repo_root, repo_root / 'malca'):
    resolved = str(path.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)

import malca.core.baseline as baseline_module
from malca.core.baseline import (
    global_median_baseline,
    per_camera_gp_baseline,
    per_camera_gp_baseline_masked,
    per_camera_median_baseline,
)
from malca.core.utils import clean_lc
from malca.io.lightcurve_io import (
    load_lightcurve_df,
    stable_camera_color,
    to_asassn_algorithm_frame,
)
from malca.stv.events import DEFAULT_BASELINE_KWARGS

JD_OFFSET = 2458000.0


In [ ]:
run_roots = [
    repo_root / 'output' / 'runs' / 'dat3-full-extended_2026-07-01-v4',
    repo_root / 'output' / 'runs' / 'runs_march18_bundle_all',
]
lightcurve_dirs = [root / 'bundle_assets' / 'lightcurves' for root in run_roots]
lightcurve_dirs = [path for path in lightcurve_dirs if path.is_dir()]
if not lightcurve_dirs:
    raise FileNotFoundError('No bundled light-curve directory was found under output/runs.')

target_ids = [
    '197569146752',  # strong corroborated-propagation stress case
    '214748665650',  # propagation plus same-band late-onset consensus
    '489626721133',
    '635656111241',
    '438086746412',
]
target_paths = {}
for target_id in target_ids:
    matches = [path / f'{target_id}.dat3' for path in lightcurve_dirs]
    target_paths[target_id] = next((path for path in matches if path.is_file()), None)

missing = [target_id for target_id, path in target_paths.items() if path is None]
if missing:
    raise FileNotFoundError(f'Missing requested bundled light curves: {missing}')

display(pd.DataFrame({
    'asas_sn_id': target_ids,
    'lightcurve_path': [str(target_paths[target_id]) for target_id in target_ids],
}))


In [ ]:
@contextmanager
def previous_masking_behavior():
    """Reconstruct the immediately previous masked-GP behavior."""
    original = baseline_module._corroborated_intervals

    def no_corroborated_intervals(*args, **kwargs):
        return []

    baseline_module._corroborated_intervals = no_corroborated_intervals
    try:
        yield
    finally:
        baseline_module._corroborated_intervals = original


def load_prepared_lightcurve(path):
    canonical = load_lightcurve_df(path, apply_quality=True)
    if canonical is None or canonical.empty:
        raise ValueError(f'No light-curve rows were loaded from {path}')
    algorithm_frame = to_asassn_algorithm_frame(canonical)
    return clean_lc(algorithm_frame).sort_values('JD').reset_index(drop=True)


def pooled_by_band_gp(lightcurve, baseline_func):
    """Run the supplied GP path once per filter instead of once per camera."""
    pooled = lightcurve.copy()
    if '_gp_partition' in pooled.columns:
        raise ValueError("Input already contains the reserved '_gp_partition' column.")
    if 'v_g_band' in pooled.columns:
        band_labels = pooled['v_g_band'].astype('string').fillna('unknown')
    else:
        band_labels = pd.Series('all', index=pooled.index, dtype='string')
    pooled['_gp_partition'] = 'pooled_band_' + band_labels
    result = baseline_func(
        pooled, cam_col='_gp_partition', **DEFAULT_BASELINE_KWARGS
    ).reset_index(drop=True)
    return result.drop(columns=['_gp_partition'])


def run_capability_ladder(lightcurve):
    results = {
        '1. Global median': global_median_baseline(lightcurve).reset_index(drop=True),
        '2. Per-camera rolling median': per_camera_median_baseline(lightcurve).reset_index(drop=True),
        '3. Unmasked per-camera GP': per_camera_gp_baseline(
            lightcurve, **DEFAULT_BASELINE_KWARGS
        ).reset_index(drop=True),
    }
    with previous_masking_behavior():
        results['4. Previous masked GP'] = per_camera_gp_baseline_masked(
            lightcurve, **DEFAULT_BASELINE_KWARGS
        ).reset_index(drop=True)
    results['5. Current masked GP + propagation'] = per_camera_gp_baseline_masked(
        lightcurve, **DEFAULT_BASELINE_KWARGS
    ).reset_index(drop=True)
    return results


def robust_scatter(values):
    values = pd.to_numeric(pd.Series(values), errors='coerce').to_numpy(float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    center = float(np.median(values))
    return float(1.4826 * np.median(np.abs(values - center)))


def summarize_ladder(target_id, lightcurve, results):
    previous = results['4. Previous masked GP']
    current = results['5. Current masked GP + propagation']
    previous_mask = previous['is_masked'].fillna(False).to_numpy(bool)
    current_mask = current['is_masked'].fillna(False).to_numpy(bool)
    rows = []
    for method, result in results.items():
        masked_points = (
            int(result['is_masked'].fillna(False).sum())
            if 'is_masked' in result.columns else 0
        )
        consensus_cameras = (
            int(result.groupby('camera#')['needs_consensus'].any().sum())
            if 'needs_consensus' in result.columns else 0
        )
        rows.append({
            'asas_sn_id': target_id,
            'points': len(lightcurve),
            'cameras': lightcurve['camera#'].nunique(),
            'method': method,
            'residual_MAD_mag': robust_scatter(result['resid']),
            'masked_points': masked_points,
            'newly_propagated_points': (
                int(np.sum(current_mask & ~previous_mask))
                if method.startswith('5.') else 0
            ),
            'consensus_cameras': consensus_cameras,
            'baseline_sources': ', '.join(
                sorted(result['baseline_source'].dropna().astype(str).unique())
            ),
        })
    return pd.DataFrame(rows)


def plot_camera_points(ax, frame, y_column, *, alpha=0.35):
    for camera, sub in frame.groupby('camera#', sort=True):
        sub = sub.sort_values('JD')
        ax.errorbar(
            sub['JD'] - JD_OFFSET, sub[y_column],
            yerr=sub['error'] if y_column == 'mag' else None,
            fmt='.', ms=3.5, lw=0.4, alpha=alpha,
            color=stable_camera_color(camera),
        )


def overlay_mask(ax, frame, mask, y_column, *, color, marker, label):
    mask = np.asarray(mask, bool)
    if not mask.any():
        return
    ax.scatter(
        frame.loc[mask, 'JD'] - JD_OFFSET, frame.loc[mask, y_column],
        s=24, marker=marker, color=color, linewidths=0.9, zorder=5, label=label,
    )


def make_ladder_figure(target_id, lightcurve, results):
    fig, axes = plt.subplots(
        len(results), 2, figsize=(18, 19), sharex='col', sharey='col'
    )
    previous = results['4. Previous masked GP']
    current = results['5. Current masked GP + propagation']
    previous_mask = previous['is_masked'].fillna(False).to_numpy(bool)
    current_mask = current['is_masked'].fillna(False).to_numpy(bool)
    retained_local = current_mask & previous_mask
    propagation_only = current_mask & ~previous_mask

    for row, (method, result) in enumerate(results.items()):
        magnitude_ax, residual_ax = axes[row]
        plot_camera_points(magnitude_ax, result, 'mag')
        plot_camera_points(residual_ax, result, 'resid')
        for camera, sub in result.groupby('camera#', sort=True):
            sub = sub.sort_values('JD')
            magnitude_ax.plot(
                sub['JD'] - JD_OFFSET, sub['baseline'],
                color=stable_camera_color(camera), lw=1.4,
            )

        if method.startswith('4.'):
            overlay_mask(
                magnitude_ax, result, previous_mask, 'mag',
                color='#d73027', marker='x', label='camera-local mask',
            )
            overlay_mask(
                residual_ax, result, previous_mask, 'resid',
                color='#d73027', marker='x', label='camera-local mask',
            )
        elif method.startswith('5.'):
            for ax, column in ((magnitude_ax, 'mag'), (residual_ax, 'resid')):
                overlay_mask(
                    ax, result, retained_local, column,
                    color='#d73027', marker='x', label='retained local mask',
                )
                overlay_mask(
                    ax, result, propagation_only, column,
                    color='#7b3294', marker='+', label='newly propagated',
                )

        residual_ax.axhline(0.0, color='0.25', lw=0.8)
        magnitude_ax.set_title(method, loc='left')
        residual_ax.set_title(f'{method} residual', loc='left')
        magnitude_ax.set_ylabel('magnitude')
        residual_ax.set_ylabel('residual [mag]')
        magnitude_ax.tick_params(direction='in', top=True, right=True)
        residual_ax.tick_params(direction='in', top=True, right=True)

    axes[0, 0].invert_yaxis()
    axes[0, 1].invert_yaxis()
    axes[-1, 0].set_xlabel('JD - 2458000')
    axes[-1, 1].set_xlabel('JD - 2458000')
    axes[-1, 0].legend(loc='best')
    axes[-1, 1].legend(loc='best')
    fig.suptitle(f'Baseline capability ladder: ASAS-SN {target_id}', fontsize=16)
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    return fig


## Successive comparison on real bundled light curves

The same cleaned observations and the same live defaults are supplied to every method. Red crosses show the previous/current camera-local mask where it remains active. Purple plus signs show points excluded only because the newest same-band corroboration step propagated an interval into that camera. The residual MAD is descriptive only: a flexible baseline can lower it by absorbing real variability, so a smaller value is not automatically a better baseline.

In [ ]:
comparison_runs = {}
summary_tables = []

for target_id in target_ids:
    lightcurve = load_prepared_lightcurve(target_paths[target_id])
    results = run_capability_ladder(lightcurve)
    comparison_runs[target_id] = {'lightcurve': lightcurve, 'results': results}
    summary_tables.append(summarize_ladder(target_id, lightcurve, results))

    figure = make_ladder_figure(target_id, lightcurve, results)
    display(figure)
    plt.close(figure)

summary = pd.concat(summary_tables, ignore_index=True)
display(summary)


## Camera-partition ablation: pooled versus per-camera GP

The pooled arm fits one GP to all cameras within each filter; it does **not** combine V and g magnitudes. The per-camera arm is the live implementation. Both receive the same cleaned rows, kernel settings, uncertainty model, and masking defaults. The only controlled change is whether `cam_col` identifies the physical camera or a synthetic filter-level group.

The comparison is repeated for the unmasked GP and the complete two-pass masked GP. In the pooled masked arm, camera-to-camera corroboration and late-onset camera consensus are naturally unavailable because there is only one fit group per filter; that is part of the complexity being ablated. Baseline differences and mask disagreements are therefore more informative than residual MAD alone.

In [ ]:
def camera_partition_results(lightcurve, ladder_results):
    return {
        'Unmasked': {
            'Pooled by band': pooled_by_band_gp(lightcurve, per_camera_gp_baseline),
            'Per camera': ladder_results['3. Unmasked per-camera GP'],
        },
        'Masked': {
            'Pooled by band': pooled_by_band_gp(lightcurve, per_camera_gp_baseline_masked),
            'Per camera': ladder_results['5. Current masked GP + propagation'],
        },
    }


def summarize_camera_partition(target_id, lightcurve, results):
    rows = []
    for gp_path, pair in results.items():
        pooled = pair['Pooled by band']
        per_camera = pair['Per camera']
        baseline_delta = (
            pd.to_numeric(per_camera['baseline'], errors='coerce').to_numpy(float)
            - pd.to_numeric(pooled['baseline'], errors='coerce').to_numpy(float)
        )
        finite_delta = np.abs(baseline_delta[np.isfinite(baseline_delta)])
        pooled_mask = (
            pooled['is_masked'].fillna(False).to_numpy(bool)
            if 'is_masked' in pooled.columns else np.zeros(len(pooled), dtype=bool)
        )
        per_camera_mask = (
            per_camera['is_masked'].fillna(False).to_numpy(bool)
            if 'is_masked' in per_camera.columns else np.zeros(len(per_camera), dtype=bool)
        )
        shared_quiet = ~(pooled_mask | per_camera_mask)
        pooled_resid_mad = robust_scatter(pooled['resid'])
        per_camera_resid_mad = robust_scatter(per_camera['resid'])
        pooled_camera_offsets = pooled.groupby('camera#', sort=True)['resid'].median()
        per_camera_offsets = per_camera.groupby('camera#', sort=True)['resid'].median()
        rows.append({
            'asas_sn_id': target_id,
            'gp_path': gp_path.lower(),
            'points': len(lightcurve),
            'bands': int(lightcurve['v_g_band'].nunique(dropna=True)),
            'cameras': int(lightcurve['camera#'].nunique(dropna=True)),
            'pooled_residual_MAD_mag': pooled_resid_mad,
            'per_camera_residual_MAD_mag': per_camera_resid_mad,
            'per_camera_minus_pooled_residual_MAD_mag': (
                per_camera_resid_mad - pooled_resid_mad
            ),
            'pooled_shared_quiet_residual_MAD_mag': (
                robust_scatter(pooled.loc[shared_quiet, 'resid'])
            ),
            'per_camera_shared_quiet_residual_MAD_mag': (
                robust_scatter(per_camera.loc[shared_quiet, 'resid'])
            ),
            'pooled_camera_offset_MAD_mag': robust_scatter(pooled_camera_offsets),
            'per_camera_camera_offset_MAD_mag': robust_scatter(per_camera_offsets),
            'median_abs_baseline_delta_mag': (
                float(np.median(finite_delta)) if finite_delta.size else np.nan
            ),
            'p95_abs_baseline_delta_mag': (
                float(np.percentile(finite_delta, 95)) if finite_delta.size else np.nan
            ),
            'pooled_masked_points': int(pooled_mask.sum()) if gp_path == 'Masked' else 0,
            'per_camera_masked_points': (
                int(per_camera_mask.sum()) if gp_path == 'Masked' else 0
            ),
            'mask_disagreement_points': (
                int(np.sum(pooled_mask != per_camera_mask)) if gp_path == 'Masked' else 0
            ),
        })
    return pd.DataFrame(rows)


def plot_partition_curve(ax, frame, *, pooled):
    group_column = 'v_g_band' if pooled and 'v_g_band' in frame.columns else 'camera#'
    for group_id, sub in frame.groupby(group_column, sort=True):
        sub = sub.sort_values('JD')
        color = (
            '#2166ac' if pooled and float(group_id) == 0.0
            else '#b2182b' if pooled
            else stable_camera_color(group_id)
        )
        ax.plot(sub['JD'] - JD_OFFSET, sub['baseline'], color=color, lw=1.35)


def make_camera_partition_figure(target_id, lightcurve, results):
    fig, axes = plt.subplots(2, 4, figsize=(24, 10), sharex=True)
    residual_values = np.concatenate([
        pd.to_numeric(result['resid'], errors='coerce').to_numpy(float)
        for pair in results.values() for result in pair.values()
    ])
    residual_values = residual_values[np.isfinite(residual_values)]
    residual_limit = max(0.15, float(np.max(np.abs(residual_values))) * 1.05)
    magnitude_values = pd.to_numeric(lightcurve['mag'], errors='coerce').to_numpy(float)
    magnitude_values = magnitude_values[np.isfinite(magnitude_values)]
    magnitude_pad = max(0.02, float(np.ptp(magnitude_values)) * 0.05)
    magnitude_limits = (
        float(np.max(magnitude_values)) + magnitude_pad,
        float(np.min(magnitude_values)) - magnitude_pad,
    )

    for row, (gp_path, pair) in enumerate(results.items()):
        pooled = pair['Pooled by band']
        per_camera = pair['Per camera']
        panels = (
            (axes[row, 0], pooled, 'mag', True, 'Pooled by band: baseline'),
            (axes[row, 1], per_camera, 'mag', False, 'Per camera: baseline'),
            (axes[row, 2], pooled, 'resid', True, 'Pooled by band: residual'),
            (axes[row, 3], per_camera, 'resid', False, 'Per camera: residual'),
        )
        for ax, frame, column, pooled_arm, title in panels:
            plot_camera_points(ax, frame, column)
            if column == 'mag':
                plot_partition_curve(ax, frame, pooled=pooled_arm)
                ax.set_ylim(*magnitude_limits)
            else:
                ax.axhline(0.0, color='0.25', lw=0.8)
                ax.set_ylim(residual_limit, -residual_limit)
            if gp_path == 'Masked':
                mask = frame['is_masked'].fillna(False).to_numpy(bool)
                overlay_mask(
                    ax, frame, mask, column,
                    color='#7b3294' if pooled_arm else '#d73027',
                    marker='x', label='masked by this arm',
                )
            ax.set_title(title, loc='left')
            ax.tick_params(direction='in', top=True, right=True)
        axes[row, 0].set_ylabel(f'{gp_path} GP\nmagnitude')
        axes[row, 2].set_ylabel(f'{gp_path} GP\nresidual [mag]')

    for ax in axes[-1]:
        ax.set_xlabel('JD - 2458000')
    axes[1, 0].legend(loc='best')
    axes[1, 1].legend(loc='best')
    axes[1, 2].legend(loc='best')
    axes[1, 3].legend(loc='best')
    fig.suptitle(
        f'Camera-partition ablation: ASAS-SN {target_id}', fontsize=16
    )
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    return fig


camera_partition_runs = {}
camera_partition_tables = []
for target_id in target_ids:
    run = comparison_runs[target_id]
    results = camera_partition_results(run['lightcurve'], run['results'])
    camera_partition_runs[target_id] = results
    camera_partition_tables.append(
        summarize_camera_partition(target_id, run['lightcurve'], results)
    )
    figure = make_camera_partition_figure(
        target_id, run['lightcurve'], results
    )
    display(figure)
    plt.close(figure)

camera_partition_summary = pd.concat(camera_partition_tables, ignore_index=True)
display(camera_partition_summary)


## Parallel masked-GP branches: non-per-camera versus per-camera

Every stage is now shown as an adjacent pair: the left panel pools all cameras within each filter, while the right panel follows the production per-camera branch. The layout therefore carries the camera-partition ablation through the stiff GP, initial masking, corroborated propagation, late-onset consensus, final baseline, and standardized residual rather than comparing only the final products.

By default this section displays 24 reviewed Dippers, not only the two original examples. `197569146752` and `214748665650` are always included as known propagation/consensus stress cases; the other sources are selected deterministically across the sorted live reviewed-Dipper cohort after requiring a bundled light curve, at least 200 cleaned epochs, and at least two cameras. Change `STAGE_COMPARISON_SOURCE_COUNT` to widen or narrow the displayed cohort.

In [ ]:
STAGE_COMPARISON_SOURCE_COUNT = 24
STAGE_REQUIRED_SOURCE_IDS = ('197569146752', '214748665650')
STAGE_MIN_POINTS = 200
STAGE_MIN_CAMERAS = 2


def resolve_bundled_lightcurve(source_id):
    matches = [path / f'{source_id}.dat3' for path in lightcurve_dirs]
    return next((path for path in matches if path.is_file()), None)


def select_stage_sources(source_count=STAGE_COMPARISON_SOURCE_COUNT):
    review_db = run_roots[0] / 'review' / 'review.db'
    query = """
        SELECT DISTINCT trim(c.asas_sn_id) AS asas_sn_id
        FROM candidates AS c
        INNER JOIN reviews AS r ON r.candidate_id = c.candidate_id
        WHERE lower(trim(coalesce(r.event_class, ''))) = 'dipper'
          AND trim(coalesce(c.asas_sn_id, '')) <> ''
        ORDER BY asas_sn_id
    """
    read_only_uri = f'file:{review_db.resolve().as_posix()}?mode=ro'
    with sqlite3.connect(read_only_uri, uri=True) as connection:
        cohort = pd.read_sql_query(query, connection)

    metadata_rows = []
    for raw_source_id in cohort['asas_sn_id']:
        source_id = str(raw_source_id).strip()
        path = resolve_bundled_lightcurve(source_id)
        if path is None:
            continue
        try:
            lightcurve = load_prepared_lightcurve(path)
        except Exception:
            continue
        metadata_rows.append({
            'asas_sn_id': source_id,
            'lightcurve_path': str(path),
            'points': len(lightcurve),
            'cameras': int(lightcurve['camera#'].nunique(dropna=True)),
            'bands': int(lightcurve['v_g_band'].nunique(dropna=True)),
        })

    inventory = pd.DataFrame(metadata_rows).sort_values('asas_sn_id').reset_index(drop=True)
    eligible = inventory.loc[
        (inventory['points'] >= STAGE_MIN_POINTS)
        & (inventory['cameras'] >= STAGE_MIN_CAMERAS)
    ].reset_index(drop=True)
    eligible_ids = set(eligible['asas_sn_id'])
    selected_ids = [
        source_id for source_id in STAGE_REQUIRED_SOURCE_IDS
        if source_id in eligible_ids
    ]
    remaining = eligible.loc[~eligible['asas_sn_id'].isin(selected_ids)].reset_index(drop=True)
    extra_count = min(
        max(int(source_count) - len(selected_ids), 0), len(remaining)
    )
    if extra_count:
        extra_indices = np.linspace(0, len(remaining) - 1, extra_count, dtype=int)
        selected_ids.extend(remaining.iloc[extra_indices]['asas_sn_id'].tolist())
    selected = (
        eligible.set_index('asas_sn_id').loc[selected_ids].reset_index()
    )
    selected.insert(0, 'display_order', np.arange(1, len(selected) + 1))
    return selected, inventory


stage_source_selection, stage_source_inventory = select_stage_sources()
display(stage_source_selection)


def plot_stage_data(ax, frame, y_column):
    plot_camera_points(ax, frame, y_column, alpha=0.32)
    ax.tick_params(direction='in', top=True, right=True)


def plot_camera_curve(ax, frame, column, *, linestyle='-', linewidth=1.4):
    for camera, sub in frame.groupby('camera#', sort=True):
        sub = sub.sort_values('JD')
        values = pd.to_numeric(sub[column], errors='coerce').to_numpy(float)
        if np.isfinite(values).any():
            ax.plot(
                sub['JD'] - JD_OFFSET, values,
                linestyle=linestyle, lw=linewidth, color=stable_camera_color(camera),
            )


def plot_branch_curve(
    ax, frame, column, *, pooled, linestyle='-', linewidth=1.4
):
    group_column = 'v_g_band' if pooled else 'camera#'
    for group_id, sub in frame.groupby(group_column, sort=True):
        sub = sub.sort_values('JD')
        values = pd.to_numeric(sub[column], errors='coerce').to_numpy(float)
        if not np.isfinite(values).any():
            continue
        if pooled:
            band_value = pd.to_numeric(pd.Series([group_id]), errors='coerce').iloc[0]
            color = '#2166ac' if np.isclose(band_value, 0.0) else '#b2182b'
        else:
            color = stable_camera_color(group_id)
        ax.plot(
            sub['JD'] - JD_OFFSET, values, linestyle=linestyle,
            lw=linewidth, color=color,
        )


STAGE_TITLES = {
    1: 'Stiff GP (base_rough)',
    2: 'Initial branch-local masking',
    3: 'Corroborated same-band propagation',
    4: 'Late-onset consensus',
    5: 'Final quiescent baseline',
    6: 'Standardized residual using sigma_eff',
}


def draw_branch_stage(
    ax, stage, branch_label, previous, current, *, pooled
):
    previous_mask = previous['is_masked'].fillna(False).to_numpy(bool)
    current_mask = current['is_masked'].fillna(False).to_numpy(bool)
    retained_local = current_mask & previous_mask
    propagation_only = current_mask & ~previous_mask

    if stage == 1:
        plot_stage_data(ax, current, 'mag')
        plot_branch_curve(ax, current, 'base_rough', pooled=pooled, linestyle=':')
    elif stage == 2:
        plot_stage_data(ax, previous, 'mag')
        plot_branch_curve(ax, previous, 'baseline', pooled=pooled)
        overlay_mask(
            ax, previous, previous_mask, 'mag', color='#d73027', marker='x',
            label='pooled-band mask' if pooled else 'camera-local mask',
        )
    elif stage == 3:
        plot_stage_data(ax, current, 'mag')
        plot_branch_curve(ax, current, 'baseline', pooled=pooled)
        overlay_mask(
            ax, current, retained_local, 'mag', color='#d73027', marker='x',
            label='retained branch-local mask',
        )
        overlay_mask(
            ax, current, propagation_only, 'mag', color='#7b3294', marker='+',
            label='newly propagated',
        )
        if pooled and not propagation_only.any():
            ax.text(
                0.5, 0.08, 'No camera-to-camera propagation',
                transform=ax.transAxes, ha='center', va='bottom',
            )
    elif stage == 4:
        plot_stage_data(ax, current, 'mag')
        plot_branch_curve(
            ax, current, 'base_consensus', pooled=pooled,
            linestyle='--', linewidth=1.8,
        )
        consensus_cameras = int(
            current.groupby('camera#')['needs_consensus'].any().sum()
        )
        if consensus_cameras == 0:
            message = (
                'No separate-camera consensus'
                if pooled else 'No camera required consensus'
            )
            ax.text(
                0.5, 0.08, message, transform=ax.transAxes,
                ha='center', va='bottom',
            )
    elif stage == 5:
        plot_stage_data(ax, current, 'mag')
        plot_branch_curve(ax, current, 'baseline', pooled=pooled)
    elif stage == 6:
        plot_stage_data(ax, current, 'sigma_resid')
        ax.axhline(0, color='0.25', lw=0.8)
        ax.axhline(3, color='0.5', lw=0.8, ls='--')
        ax.axhline(-3, color='0.5', lw=0.8, ls='--')
        overlay_mask(
            ax, current, propagation_only, 'sigma_resid',
            color='#7b3294', marker='+', label='newly propagated',
        )
    else:
        raise ValueError(f'Unknown stage: {stage}')

    ax.set_title(f'{stage}. {STAGE_TITLES[stage]}\n{branch_label}', loc='left')
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc='best')


def make_stage_figure(
    target_id, lightcurve, pooled_previous, pooled_current,
    per_camera_previous, per_camera_current,
):
    fig, axes = plt.subplots(3, 4, figsize=(24, 15), sharex=True)
    branches = (
        ('Non-per-camera (pooled by band)', True, pooled_previous, pooled_current),
        ('Per camera', False, per_camera_previous, per_camera_current),
    )
    finite_mag = pd.to_numeric(lightcurve['mag'], errors='coerce').to_numpy(float)
    finite_mag = finite_mag[np.isfinite(finite_mag)]
    mag_pad = max(0.02, float(np.ptp(finite_mag)) * 0.05)
    mag_limits = (
        float(np.max(finite_mag)) + mag_pad,
        float(np.min(finite_mag)) - mag_pad,
    )
    sigma_values = np.concatenate([
        pd.to_numeric(frame['sigma_resid'], errors='coerce').to_numpy(float)
        for frame in (pooled_current, per_camera_current)
    ])
    sigma_values = sigma_values[np.isfinite(sigma_values)]
    sigma_values = np.concatenate([sigma_values, np.array([-3.0, 0.0, 3.0])])
    sigma_pad = max(0.5, float(np.ptp(sigma_values)) * 0.05)
    sigma_limits = (
        float(np.max(sigma_values)) + sigma_pad,
        float(np.min(sigma_values)) - sigma_pad,
    )

    for stage in range(1, 7):
        row = (stage - 1) // 2
        pair_start = 2 * ((stage - 1) % 2)
        for branch_offset, (label, pooled, previous, current) in enumerate(branches):
            ax = axes[row, pair_start + branch_offset]
            draw_branch_stage(
                ax, stage, label, previous, current, pooled=pooled
            )
            if stage < 6:
                ax.set_ylim(*mag_limits)
            else:
                ax.set_ylim(*sigma_limits)
        axes[row, pair_start].set_ylabel(
            'magnitude' if stage < 6 else 'standardized residual'
        )

    for ax in axes[-1]:
        ax.set_xlabel('JD - 2458000')
    fig.suptitle(
        f'Masked-GP stages: non-per-camera vs per-camera — ASAS-SN {target_id}',
        fontsize=16,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    return fig


stage_manifest_rows = []
for selection_row in stage_source_selection.itertuples(index=False):
    target_id = str(selection_row.asas_sn_id)
    print(
        f'[{selection_row.display_order:02d}/{len(stage_source_selection):02d}] '
        f'ASAS-SN {target_id}', end=' ... '
    )
    try:
        if target_id in comparison_runs:
            run = comparison_runs[target_id]
            lightcurve = run['lightcurve']
            per_camera_previous = run['results']['4. Previous masked GP']
            per_camera_current = run['results']['5. Current masked GP + propagation']
            pooled_current = camera_partition_runs[target_id]['Masked']['Pooled by band']
        else:
            lightcurve = load_prepared_lightcurve(Path(selection_row.lightcurve_path))
            with previous_masking_behavior():
                per_camera_previous = per_camera_gp_baseline_masked(
                    lightcurve, **DEFAULT_BASELINE_KWARGS
                ).reset_index(drop=True)
            per_camera_current = per_camera_gp_baseline_masked(
                lightcurve, **DEFAULT_BASELINE_KWARGS
            ).reset_index(drop=True)
            pooled_current = pooled_by_band_gp(
                lightcurve, per_camera_gp_baseline_masked
            )

        with previous_masking_behavior():
            pooled_previous = pooled_by_band_gp(
                lightcurve, per_camera_gp_baseline_masked
            )

        figure = make_stage_figure(
            target_id, lightcurve, pooled_previous, pooled_current,
            per_camera_previous, per_camera_current,
        )
        display(figure)
        plt.close(figure)

        pooled_mask = pooled_current['is_masked'].fillna(False).to_numpy(bool)
        per_camera_mask = per_camera_current['is_masked'].fillna(False).to_numpy(bool)
        stage_manifest_rows.append({
            'display_order': int(selection_row.display_order),
            'asas_sn_id': target_id,
            'points': len(lightcurve),
            'bands': int(lightcurve['v_g_band'].nunique(dropna=True)),
            'cameras': int(lightcurve['camera#'].nunique(dropna=True)),
            'pooled_masked_points': int(pooled_mask.sum()),
            'per_camera_masked_points': int(per_camera_mask.sum()),
            'mask_disagreement_points': int(np.sum(pooled_mask != per_camera_mask)),
            'per_camera_consensus_cameras': int(
                per_camera_current.groupby('camera#')['needs_consensus'].any().sum()
            ),
            'status': 'ok',
            'error': '',
        })
        print('ok')
    except Exception as exc:
        stage_manifest_rows.append({
            'display_order': int(selection_row.display_order),
            'asas_sn_id': target_id,
            'points': int(selection_row.points),
            'bands': int(selection_row.bands),
            'cameras': int(selection_row.cameras),
            'pooled_masked_points': 0,
            'per_camera_masked_points': 0,
            'mask_disagreement_points': 0,
            'per_camera_consensus_cameras': 0,
            'status': 'failed',
            'error': f'{type(exc).__name__}: {exc}',
        })
        print(f'failed: {type(exc).__name__}: {exc}')

stage_comparison_manifest = pd.DataFrame(stage_manifest_rows)
display(stage_comparison_manifest)


## Opt-in calibrated cross-band consensus

Same-band consensus is automatic for qualifying late-onset cameras. Cross-band transfer remains disabled by default and must be requested explicitly. When enabled, it requires sufficient temporal overlap, estimates the inter-band magnitude offset robustly, and records the offset, scatter, overlap count, and calibration flag rather than copying a baseline between filters without calibration. The deterministic example below makes the expected one-magnitude offset visible.

In [ ]:
def make_cross_band_example(seed=711):
    rng = np.random.default_rng(seed)
    rows = []
    dip_center = 9600.0
    for camera in ('g1', 'g2'):
        jd = np.arange(7000.0, 11000.0, 5.0)
        mag = 14.0 + rng.normal(0.0, 0.015, len(jd))
        mag += 0.45 * np.exp(-0.5 * ((jd - dip_center) / 15.0) ** 2)
        rows.extend(
            {'JD': t, 'mag': m, 'error': 0.015, 'camera#': camera,
             'v_g_band': 0, 'saturated': 0}
            for t, m in zip(jd, mag)
        )
    for camera in ('v1', 'v2'):
        jd = np.arange(9500.0, 11000.0, 5.0)
        mag = 15.0 + rng.normal(0.0, 0.015, len(jd))
        mag += 0.45 * np.exp(-0.5 * ((jd - dip_center) / 15.0) ** 2)
        rows.extend(
            {'JD': t, 'mag': m, 'error': 0.015, 'camera#': camera,
             'v_g_band': 1, 'saturated': 0}
            for t, m in zip(jd, mag)
        )
    return pd.DataFrame(rows).sort_values('JD').reset_index(drop=True)


cross_band_input = make_cross_band_example()
cross_band_input['JD'] += JD_OFFSET
without_transfer = per_camera_gp_baseline_masked(
    cross_band_input,
    late_onset_buffer_days=300.0,
    min_anchor_overlap_days=30.0,
    allow_cross_band_consensus=False,
).reset_index(drop=True)
with_transfer = per_camera_gp_baseline_masked(
    cross_band_input,
    late_onset_buffer_days=300.0,
    min_anchor_overlap_days=30.0,
    allow_cross_band_consensus=True,
    cross_band_min_overlap_points=50,
).reset_index(drop=True)

v_band = with_transfer['camera#'].astype(str).str.startswith('v')
calibration_summary = (
    with_transfer.loc[v_band, [
        'camera#', 'baseline_source', 'cross_band_calibrated',
        'cross_band_offset_mag', 'cross_band_offset_scatter',
        'cross_band_overlap_points',
    ]]
    .drop_duplicates()
    .sort_values('camera#')
)
display(calibration_summary)

assert with_transfer.loc[v_band, 'cross_band_calibrated'].all()
assert np.isclose(
    np.nanmedian(with_transfer.loc[v_band, 'cross_band_offset_mag']), 1.0, atol=0.05
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True, sharey=True)
for ax, result, title in (
    (axes[0], without_transfer, 'Cross-band transfer disabled (default)'),
    (axes[1], with_transfer, 'Calibrated cross-band consensus enabled'),
):
    selected = result['camera#'].astype(str).str.startswith('v')
    frame = result.loc[selected]
    plot_camera_points(ax, frame, 'mag', alpha=0.35)
    plot_camera_curve(ax, frame, 'baseline')
    ax.set_title(title)
    ax.set_xlabel('synthetic JD - 2458000')
    ax.set_ylabel('V-band magnitude')
    ax.tick_params(direction='in', top=True, right=True)
axes[0].invert_yaxis()
g_band = with_transfer['camera#'].astype(str).str.startswith('g')
for camera, sub in with_transfer.loc[g_band].groupby('camera#', sort=True):
    sub = sub.sort_values('JD')
    axes[1].plot(
        sub['JD'] - JD_OFFSET, sub['baseline'],
        color='0.35', lw=1.0, ls=':',
        label='unshifted g-band anchor' if camera == 'g1' else None,
    )
offset = float(np.nanmedian(with_transfer.loc[v_band, 'cross_band_offset_mag']))
axes[1].text(
    0.98, 0.06, f'calibrated offset = {offset:+.3f} mag',
    transform=axes[1].transAxes, ha='right', va='bottom',
)
axes[1].legend(loc='center right')
fig.tight_layout()
display(fig)
plt.close(fig)
